In [1]:
import polars as pl
import numpy as np
import pandas as pd
import sys
import lightgbm as lgb
import xgboost as xgb
import optuna
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from pathlib import Path

sys.path.append(str(Path("..").resolve()))
from src.features.feature_registry import get_feature_cols, get_categorical_cols, get_binary_cols, get_numeric_cols, TARGET_COL, ID_COL

In [2]:
PROCESSED_DIR = Path("..") / "data" / "processed"

# Sort by SK_ID_CURR (sequential application ID — temporal proxy for this dataset)
train = pl.read_parquet(PROCESSED_DIR / "train_complete").sort(ID_COL)
test  = pl.read_parquet(PROCESSED_DIR / "test_complete")

FEATURES = get_feature_cols()

X = train[FEATURES].to_pandas()
y = train[TARGET_COL].to_numpy()

X_test = test[FEATURES].to_pandas()

n = y.sum() / len(y)
n

np.float64(0.08072881945686496)

In [3]:
len(y)

307511

In [4]:
cat_cols = get_categorical_cols()
num_cols = get_numeric_cols() + get_binary_cols()

cat_idx = [FEATURES.index(c) for c in cat_cols if c in FEATURES]
num_idx = [FEATURES.index(c) for c in num_cols if c in FEATURES]

preprocessor = ColumnTransformer([
    ("num",StandardScaler(),num_idx),
    ("cat",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1),cat_idx)
])

In [5]:
tscv = TimeSeriesSplit(n_splits=5)

In [6]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42))
])

scores = cross_val_score(model, X, y, cv=tscv, scoring="roc_auc")
print("ROC-AUC per fold:", scores)
print("Mean ROC-AUC:    ", scores.mean())
print("Std:             ", scores.std())

ROC-AUC per fold: [0.75795295 0.76222037 0.7557146  0.77061103 0.76843165]
Mean ROC-AUC:     0.7629861195084124
Std:              0.0057721255337856


In [7]:
X_encoded = X.copy()
X_test_encoded = X_test.copy()

print(type(X_encoded))
print(type(X))

for c in cat_cols:
    X_encoded[c] = X_encoded[c].astype("category").cat.codes
    X_test_encoded[c] = X_test_encoded[c].astype("category").cat.codes

X_np = X_encoded.to_numpy()
y_np = y

<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>


In [8]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    scale_pos_weight=11,
    random_state=42,
    n_jobs=-1
)

scores_lgb = cross_val_score(lgb_model, X_np, y_np, cv=tscv, scoring="roc_auc")
print("ROC-AUC per fold:", scores_lgb)
print("Mean ROC-AUC:    ", scores_lgb.mean())
print("Std:             ", scores_lgb.std())

[LightGBM] [Info] Number of positive: 4148, number of negative: 47108
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.057099 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 20940
[LightGBM] [Info] Number of data points in the train set: 51256, number of used features: 174
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080927 -> initscore=-2.429817
[LightGBM] [Info] Start training from score -2.429817


d:\modern-data-science-tech-stack\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 8321, number of negative: 94186
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.059219 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21771
[LightGBM] [Info] Number of data points in the train set: 102507, number of used features: 177
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.081175 -> initscore=-2.426489
[LightGBM] [Info] Start training from score -2.426489


d:\modern-data-science-tech-stack\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 12542, number of negative: 141216
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.077249 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21976
[LightGBM] [Info] Number of data points in the train set: 153758, number of used features: 177
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.081570 -> initscore=-2.421208
[LightGBM] [Info] Start training from score -2.421208


...

[LightGBM] [Info] Number of positive: 16643, number of negative: 188366
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.115195 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21963
[LightGBM] [Info] Number of data points in the train set: 205009, number of used features: 179
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.081182 -> initscore=-2.426397
[LightGBM] [Info] Start training from score -2.426397


d:\modern-data-science-tech-stack\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 20764, number of negative: 235496
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.149763 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21968
[LightGBM] [Info] Number of data points in the train set: 256260, number of used features: 179
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.081027 -> initscore=-2.428473
[LightGBM] [Info] Start training from score -2.428473


d:\modern-data-science-tech-stack\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


ROC-AUC per fold: [0.74662736 0.76596672 0.76552932 0.77841798 0.78311511]
Mean ROC-AUC:     0.767931297623498
Std:              0.01268073820563133


In [9]:
xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=50,
    scale_pos_weight=11,
    random_state=42,
    n_jobs=-1
)

scores_xgb = cross_val_score(xgb_model, X_np, y_np, cv=tscv, scoring="roc_auc")
print("ROC-AUC per fold:", scores_xgb)
print("Mean ROC-AUC:    ", scores_xgb.mean())
print("Std:             ", scores_xgb.std())

ROC-AUC per fold: [0.75259229 0.76640626 0.76857541 0.78091263 0.78234462]
Mean ROC-AUC:     0.7701662410973359
Std:              0.010856426622577657


In [10]:
print(f"X shape: {X.shape}")        # should be (307511, ~80+)
print(f"Features count: {len(FEATURES)}")
print(f"Any NaN in X: {np.isnan(X_np).any()}")

X shape: (307511, 185)
Features count: 185
Any NaN in X: False


In [11]:
# Check what's actually in the registry vs what's in the dataframe
registered = get_feature_cols()
print(f"Registry feature count: {len(registered)}")

# Check which registered features are missing from train
missing_from_train = [f for f in registered if f not in train.columns]
print(f"Missing from train: {len(missing_from_train)}")
print(missing_from_train)

# Check train columns that aren't in registry
train_cols = set(train.columns) - {TARGET_COL, ID_COL}
unregistered = [c for c in train_cols if c not in registered]
print(f"Train columns not in registry: {len(unregistered)}")
print(unregistered)

Registry feature count: 185
Missing from train: 0
[]
Train columns not in registry: 5
['HAS_INSTALMENT_HISTORY', 'HAS_PREVIOUS_APPLICATION', 'HAS_CREDIT_CARD_HISTORY', 'HAS_BUREAU_RECORD', 'HAS_POS_HISTORY']


In [12]:
def objective(trial):

    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
        "gamma": trial.suggest_float("gamma", 0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),

        "subsample": 0.8,
        "colsample_bytree": 0.8,

        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "n_estimators": 500,
        "n_jobs": 1,
        "random_state": 42
    }

    aucs = []

    for train_idx, val_idx in tscv.split(X_np, y_np):

        X_train, X_val = X_np[train_idx], X_np[val_idx]
        y_train, y_val = y_np[train_idx], y_np[val_idx]

        model = xgb.XGBClassifier(**params)

        model.fit(X_train, y_train)

        preds = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, preds)
        aucs.append(auc)

    return np.mean(aucs)

In [13]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)   # adjust 30–100 depending on time

[I 2026-06-12 21:37:11,375] A new study created in memory with name: no-name-bff28571-e938-4c4a-8388-671894b0a468
[I 2026-06-12 21:38:53,984] Trial 0 finished with value: 0.7781926868585635 and parameters: {'max_depth': 4, 'min_child_weight': 24, 'gamma': 0.8734335387321396, 'reg_lambda': 10.19812885705423, 'scale_pos_weight': 3.1730925803983188, 'learning_rate': 0.08815909409987878}. Best is trial 0 with value: 0.7781926868585635.
[I 2026-06-12 21:40:54,780] Trial 1 finished with value: 0.7781823675172275 and parameters: {'max_depth': 5, 'min_child_weight': 6, 'gamma': 0.8114090655447496, 'reg_lambda': 0.28578413026003086, 'scale_pos_weight': 4.045467345124688, 'learning_rate': 0.03415075935621466}. Best is trial 0 with value: 0.7781926868585635.
[I 2026-06-12 21:42:53,636] Trial 2 finished with value: 0.7794052701950248 and parameters: {'max_depth': 5, 'min_child_weight': 13, 'gamma': 0.0454597098322157, 'reg_lambda': 3.7795905351369186, 'scale_pos_weight': 2.0654060136991848, 'learn

In [14]:
print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

Best AUC: 0.7798919534836778
Best params: {'max_depth': 7, 'min_child_weight': 29, 'gamma': 0.6169314425646689, 'reg_lambda': 11.87141146615091, 'scale_pos_weight': 1.9527607821398436, 'learning_rate': 0.025480623112898167}


In [15]:
# Temporal 80/20 split — holds out the most recent 20% of applications by SK_ID_CURR
split_idx = int(len(X_np) * 0.8)
X_train, X_val = X_np[:split_idx], X_np[split_idx:]
y_train, y_val = y_np[:split_idx], y_np[split_idx:]

In [16]:
final_model = xgb.XGBClassifier(
    max_depth=5,
    min_child_weight=38,
    gamma=0.16691306730017352,
    reg_lambda=16.307675156635266,
    scale_pos_weight=5.586941175444759,
    learning_rate=0.06296893710250466,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_estimators=2000,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=1,
    random_state=42,
    early_stopping_rounds=50,
)

In [17]:
final_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=200
)

[0]	validation_0-auc:0.71120
[200]	validation_0-auc:0.78274
[400]	validation_0-auc:0.78711
[600]	validation_0-auc:0.78846
[605]	validation_0-auc:0.78849


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fro

In [18]:
val_pred = final_model.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, val_pred)

0.7885152731957692

In [19]:
test_pred = final_model.predict_proba(X_test_encoded.to_numpy())[:, 1]
preds_df = pd.DataFrame(test_pred, columns=['Predicted_Probability'])

print(preds_df.head())
print(preds_df.describe())

   Predicted_Probability
0               0.137276
1               0.450120
2               0.124317
3               0.172326
4               0.451727
       Predicted_Probability
count           48744.000000
mean                0.243876
std                 0.181217
min                 0.005830
25%                 0.101549
50%                 0.192129
75%                 0.343327
max                 0.954229


In [20]:
preds_df['Predicted_Class'] = (preds_df['Predicted_Probability'] >= 0.5).astype(int)
print(preds_df['Predicted_Class'].value_counts())

Predicted_Class
0    43346
1     5398
Name: count, dtype: int64


In [21]:
def objective_lgb(trial):
    max_depth = trial.suggest_int("max_depth", 3, 7)
    max_leaves = 2 ** max_depth
    num_leaves = trial.suggest_int("num_leaves", 7, min(127, max_leaves))

    params = {
        "max_depth":        max_depth,
        "num_leaves":       num_leaves,
        "min_child_samples":trial.suggest_int("min_child_samples", 20, 100),
        "min_split_gain":   trial.suggest_float("min_split_gain", 0.0, 1.0),
        "reg_lambda":       trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
        "reg_alpha":        trial.suggest_float("reg_alpha", 0.0, 5.0),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.1),
        "scale_pos_weight": 11,
        "objective":        "binary",
        "metric":           "auc",
        "n_estimators":     2000,
        "n_jobs":           -1,
        "random_state":     42,
        "verbose":          -1,
    }

    aucs = []

    for train_idx, val_idx in tscv.split(X_encoded, y):
        X_train, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = lgb.LGBMClassifier(**params)

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=-1),
            ]
        )

        preds = model.predict_proba(X_val)[:, 1]
        aucs.append(roc_auc_score(y_val, preds))

    return np.mean(aucs)

In [22]:
study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=30)   # adjust 30–100 depending on time

[I 2026-06-12 23:04:41,788] A new study created in memory with name: no-name-26af20e5-bcf6-4061-9529-0fab7c33ff1a
[I 2026-06-12 23:05:28,556] Trial 0 finished with value: 0.7772000451712087 and parameters: {'max_depth': 5, 'num_leaves': 22, 'min_child_samples': 87, 'min_split_gain': 0.7059650488468064, 'reg_lambda': 0.23224015472988024, 'reg_alpha': 0.6911456404372796, 'subsample': 0.8602444967268998, 'colsample_bytree': 0.6848313058212154, 'learning_rate': 0.044674331643624464}. Best is trial 0 with value: 0.7772000451712087.
[I 2026-06-12 23:08:34,953] Trial 1 finished with value: 0.7781905863364316 and parameters: {'max_depth': 6, 'num_leaves': 22, 'min_child_samples': 25, 'min_split_gain': 0.42290373608115683, 'reg_lambda': 1.9718344314304528, 'reg_alpha': 3.505617463948102, 'subsample': 0.6881734442481783, 'colsample_bytree': 0.8107334017887735, 'learning_rate': 0.013158379708866765}. Best is trial 1 with value: 0.7781905863364316.
[I 2026-06-12 23:09:39,985] Trial 2 finished with

In [23]:
print("Best AUC:", study_lgb.best_value)
print("Best params:", study_lgb.best_params)

Best AUC: 0.7783211547238407
Best params: {'max_depth': 5, 'num_leaves': 13, 'min_child_samples': 22, 'min_split_gain': 0.331583082301396, 'reg_lambda': 3.134911907083484, 'reg_alpha': 3.760341394333307, 'subsample': 0.6896262994302365, 'colsample_bytree': 0.9360810526475307, 'learning_rate': 0.03673779604911607}


In [24]:
best_params = study_lgb.best_params

final_model_lgb = lgb.LGBMClassifier(
    **best_params,
    objective="binary",
    eval_metric="auc",
    n_estimators=2000,
    n_jobs=1,
    random_state=42,
    early_stopping_rounds=50,
)

In [25]:
final_model_lgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.250584
[100]	valid_0's binary_logloss: 0.244045
[150]	valid_0's binary_logloss: 0.241139
[200]	valid_0's binary_logloss: 0.239383
[250]	valid_0's binary_logloss: 0.2383
[300]	valid_0's binary_logloss: 0.237484
[350]	valid_0's binary_logloss: 0.236908
[400]	valid_0's binary_logloss: 0.236417
[450]	valid_0's binary_logloss: 0.23607
[500]	valid_0's binary_logloss: 0.235794
[550]	valid_0's binary_logloss: 0.235588
[600]	valid_0's binary_logloss: 0.235457
[650]	valid_0's binary_logloss: 0.235289
[700]	valid_0's binary_logloss: 0.235202
[750]	valid_0's binary_logloss: 0.235089
[800]	valid_0's binary_logloss: 0.235013
[850]	valid_0's binary_logloss: 0.234947
[900]	valid_0's binary_logloss: 0.234935
[950]	valid_0's binary_logloss: 0.234873
[1000]	valid_0's binary_logloss: 0.234823
[1050]	valid_0's binary_logloss: 0.234774
[1100]	valid_0's bi

,boosting_type,'gbdt'
,num_leaves,13
,max_depth,5
,learning_rate,0.03673779604911607
,n_estimators,2000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.331583082301396
,min_child_weight,0.001
,min_child_samples,22


In [26]:
val_pred = final_model_lgb.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, val_pred)

d:\modern-data-science-tech-stack\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.7883551063093266

In [27]:
test_pred = final_model_lgb.predict_proba(X_test_encoded.to_numpy())[:, 1]
preds_df = pd.DataFrame(test_pred, columns=['Predicted_Probability'])

print(preds_df.head())
print(preds_df.describe())

d:\modern-data-science-tech-stack\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   Predicted_Probability
0               0.044037
1               0.127180
2               0.028691
3               0.034448
4               0.139790
       Predicted_Probability
count           48744.000000
mean                0.070219
std                 0.078323
min                 0.001859
25%                 0.021926
50%                 0.042817
75%                 0.086879
max                 0.798754


In [28]:
preds_df['Predicted_Class'] = (preds_df['Predicted_Probability'] >= 0.5).astype(int)
print(preds_df['Predicted_Class'].value_counts())

Predicted_Class
0    48593
1      151
Name: count, dtype: int64
